# H3: does LoRA fine-tuning on diffusion counterfactuals transfer to real films?

Evaluation of H3 as pre-registered (PLAN.md, section 4, committed before any training) with its
clarifications of 2026-10-01, one of them added during the training session (the per-seed condition
on E's edited-film gain for "H3.1 supported"): MedGemma 4B trained on real films (R), RadEdit edits
(E) or both (RE), 3 seeds each, against the base model.
Scores: float32 log-odds d of the trained question "Is there cardiomegaly in this image?" (P(yes) =
sigmoid(d)), from `python -m cxr.lora evaluate`; label: segmenter CTR > 0.5.

- Primary metrics: real-film CTR-AUROC (gain = arm - base; bar = 0.2 x (1 - base)); edited-film
  Brier for H3.1's first clause (gain = base - arm; bar = 0.2 x base). A gain "moves" the model if
  it reaches the bar with a CI lower bound above 0; it "does not move" it if its CI upper bound is
  below the bar.
- Statistics: each arm's metric is the mean over its seeds; 95% CIs from 1,000 paired bootstrap
  resamples of patients (the same patients for every model); a verdict also needs the deciding
  contrast to have the same sign for each seed k (seed k of every arm).
- The primary evaluation is the fresh test set (one final run of this committed notebook); the
  test split is secondary and exploratory. While some arms have no scores yet, they are replaced
  by the base model and the notebook says DRY RUN: that only checks the code path.

In [1]:
import sys
from pathlib import Path

import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent  # the notebook runs from notebooks/
R = ROOT / "results"
SET = "fresh"
ARMS, SEEDS = ["R", "E", "RE"], [0, 1, 2]
names = ["base"] + [f"{a}{s}" for a in ARMS for s in SEEDS] + [f"E{s}real" for s in SEEDS]
measures = pd.read_csv(R / f"h3_measures_{SET}.csv")
scores, scores_info, missing = {}, {}, []
for name in names:
    path = R / f"h3_scores_{name}_{SET}.csv"
    if path.exists():
        scores[name] = pd.read_csv(path)
        info = path.with_suffix(".json")
        scores_info[name] = json.loads(info.read_text()) if info.exists() else {}
    else:
        missing.append(name)
DRY_RUN = bool(missing)
for name in missing:  # dry run: a missing model is replaced by the base model (all gains 0)
    scores[name] = scores["base"]
print(f"Set: {SET}; models scored: {len(names) - len(missing)} of {len(names)}" + (f"; DRY RUN, replaced by base: {missing}" if DRY_RUN else ""))
revisions = {i.get("revision") for i in scores_info.values() if i.get("revision")}
assert len(revisions) <= 1, f"models scored at different MedGemma revisions: {revisions}"
for arm in ARMS:  # every seed of an arm trained on the same draw of examples
    draws = {scores_info[f"{arm}{s}"].get("train_draw") for s in SEEDS if f"{arm}{s}" not in missing}
    assert len(draws) <= 1, f"{arm}: seeds trained on different draws"
print("MedGemma revision:", revisions or "not recorded", "| code commits:", {i.get("commit") for i in scores_info.values()})
print(measures.groupby("kind").size().to_dict())

Set: fresh; models scored: 13 of 13
MedGemma revision: {'290cda5eeccbee130f987c4ad74a59ae6f196408'} | code commits: {'8a2b1104294d55a6fef7ce5d80cfe4bae3a53de8'}
{'edit': 600, 'real': 2335, 'sham': 100}


## Per-image tables

In [2]:
keys = ["kind", "image", "growth"]
def table(name, question="cardiomegaly_0"):
    """One row per image for one model and question, with the measures (segmenter and CheXmask CTR,
    NIH labels): label = segmenter CTR > 0.5."""
    s = scores[name]
    s = s[s["question"] == question].merge(measures.drop(columns="patient_id"), on=keys, how="left")
    s["label"] = (s["ctr"] > 0.5).astype(float).where(s["ctr"].notna())
    return s.reset_index(drop=True)

tables = {name: table(name) for name in names}
base = tables["base"]
real_idx = base.index[(base["kind"] == "real") & base["ctr"].notna()]
edit_idx = base.index[(base["kind"] == "edit") & base["ctr"].notna()]
for name in names:  # every model is scored on the same images in the same order
    assert (tables[name][keys + ["patient_id"]].fillna(-1).values == base[keys + ["patient_id"]].fillna(-1).values).all(), name
print(f"real films with a CTR: {len(real_idx)} ({base.loc[real_idx, 'patient_id'].nunique()} patients); "
      f"edits with a CTR: {len(edit_idx)} ({base.loc[edit_idx, 'patient_id'].nunique()} patients)")

real films with a CTR: 2332 (1186 patients); edits with a CTR: 600 (100 patients)


## Metrics per model (point estimates)

In [3]:
prior = pd.read_csv(R / "h3_val_prior.csv").set_index("images")["share_ctr_above_05"]
pi_real, pi_edit = prior["real films"], prior["additions and shams"]  # val split (cxr.lora evaluate)
logit = lambda p: np.log(p / (1 - p))
print(f"prior correction: pi = {pi_real:.3f} (real films, also used for blanks), {pi_edit:.3f} (edits and shams); trained arms only")

def switch(d):
    """CTR where a logistic curve of P(yes) on CTR (P(yes) as a soft label) crosses 0.5, and its slope."""
    x = np.concatenate([d["ctr"], d["ctr"]])[:, None]
    y = np.concatenate([np.ones(len(d)), np.zeros(len(d))])
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(x, y, sample_weight=np.concatenate([d["p_yes"], 1 - d["p_yes"]]))
    return -lr.intercept_[0] / lr.coef_[0, 0], lr.coef_[0, 0]

def within_growth(d):
    num = den = 0.0
    for _, g in d.groupby("growth"):
        if 0 < g["label"].sum() < len(g):
            w = g["label"].sum() * (len(g) - g["label"].sum())
            num, den = num + w * roc_auc_score(g["label"], g["log_odds"]), den + w
    return num / den

def metrics(name, without_margin=False):
    """Every planned metric of one model. without_margin=True leaves out the images whose CTR is within
    0.025 of 0.5 (as in training): "every metric is also reported without them" (PLAN.md, H3)."""
    t, t1, eff = tables[name], table(name, "cardiomegaly_1"), table(name, "effusion_0")
    assert (t1[keys].fillna(-1).values == t[keys].fillna(-1).values).all(), "the two phrasings' rows are not aligned"
    keep = ~t["ctr"].between(0.475, 0.525) if without_margin else pd.Series(True, index=t.index)  # NaN kept
    r, q1, e = t.loc[real_idx][keep[real_idx]], t1.loc[real_idx][keep[real_idx]], t.loc[edit_idx][keep[edit_idx]]
    sham, blank = t[(t["kind"] == "sham") & keep], t[t["kind"] == "blank"]
    elab = eff[(eff["kind"] == "real") & eff["image"].isin(r["image"]) & ((eff["effusion"] == 1) | (eff["no_finding"] == 1))]
    out = {"model": name}
    for prefix, d in [("real", r), ("transfer_phrasing", q1)]:  # the real-film metrics, for both phrasings
        lab = d[(d["cardiomegaly"] == 1) | (d["no_finding"] == 1)]
        out[f"{prefix}_ctr_auroc"] = roc_auc_score(d["label"], d["log_odds"])
        out[f"{prefix}_brier"] = ((d["p_yes"] - d["label"]) ** 2).mean()
        out[f"{prefix}_nih_auroc"] = roc_auc_score(lab["cardiomegaly"], lab["log_odds"])
        out[f"{prefix}_median_yes_no_mass"] = d["yes_no_mass"].median()
        out[f"{prefix}_switch_ctr"], out[f"{prefix}_switch_slope"] = switch(d)
        if name != "base":  # also with the training prior removed, d + logit(pi): trained arms only
            p = 1 / (1 + np.exp(-(d["log_odds"] + logit(pi_real))))
            out[f"{prefix}_brier_prior_corrected"] = ((p - d["label"]) ** 2).mean()
            out[f"{prefix}_switch_ctr_prior_corrected"] = switch(d.assign(p_yes=p))[0]
    out.update({"edited_brier": ((e["p_yes"] - e["label"]) ** 2).mean(), "edited_ctr_auroc": roc_auc_score(e["label"], e["log_odds"]),
                "edited_within_growth_auroc": within_growth(e),
                "edited_yes_when_ctr_below_05": (e.loc[e["label"] == 0, "log_odds"] > 0).mean(),
                "sham_yes": (sham["log_odds"] > 0).mean(), "blank_yes": (blank["log_odds"] > 0).mean(),
                "effusion_auroc": roc_auc_score(elab["effusion"], elab["log_odds"])})
    if name != "base":
        d_edit = e["log_odds"] + logit(pi_edit)
        out["edited_brier_prior_corrected"] = ((1 / (1 + np.exp(-d_edit)) - e["label"]) ** 2).mean()
        out["edited_yes_when_ctr_below_05_prior_corrected"] = (d_edit[e["label"] == 0] > 0).mean()
        out["sham_yes_prior_corrected"] = (sham["log_odds"] + logit(pi_edit) > 0).mean()
        out["blank_yes_prior_corrected"] = (blank["log_odds"] + logit(pi_real) > 0).mean()
    for g, d in e.groupby("growth"):
        out[f"edited_median_p_yes_growth_{g:.2f}"] = d["p_yes"].median()
    return out

arm_of = {"base": "base", **{f"{a}{s}": a for a in ARMS for s in SEEDS}, **{f"E{s}real": "E, best on real val" for s in SEEDS}}
point = pd.DataFrame([metrics(n) for n in names]).set_index("model")
point_without_margin = pd.DataFrame([metrics(n, without_margin=True) for n in names]).set_index("model")
def by_arm(table):
    """Each arm's metric: the mean over its seeds, with the range of the per-seed values next to it."""
    out = table.groupby(table.index.map(arm_of), sort=False).agg(["mean", "min", "max"]).T
    return out.rename_axis(["metric", "stat"])

arms, arms_without_margin = by_arm(point), by_arm(point_without_margin)
dropped = {k: int(n) for k, n in tables["base"].loc[tables["base"]["ctr"].between(0.475, 0.525), "kind"].value_counts().items()}
print("images within 0.025 of CTR 0.5, left out of the without-margin tables:", dropped)
if not DRY_RUN:  # in a dry run the arms are copies of the base: no table goes to results/
    point.to_csv(R / f"h3_metrics_{SET}.csv")
    point_without_margin.to_csv(R / f"h3_metrics_{SET}_without_margin.csv")
    arms.to_csv(R / f"h3_arms_{SET}.csv")
    arms_without_margin.to_csv(R / f"h3_arms_{SET}_without_margin.csv")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(point.round(3).T)  # every model
    display(arms.round(3))  # every arm: mean and range over its seeds

prior correction: pi = 0.210 (real films, also used for blanks), 0.411 (edits and shams); trained arms only


images within 0.025 of CTR 0.5, left out of the without-margin tables: {'real': 493, 'edit': 188, 'sham': 14}


model,base,R0,R1,R2,E0,E1,E2,RE0,RE1,RE2,E0real,E1real,E2real
real_ctr_auroc,0.900,0.937,0.934,0.940,0.920,0.927,0.927,0.937,0.937,0.936,0.922,0.925,0.927
real_brier,0.145,0.114,0.119,0.124,0.091,0.087,0.083,0.089,0.141,0.126,0.087,0.087,0.082
real_nih_auroc,0.931,0.938,0.936,0.932,0.921,0.934,0.936,0.935,0.932,0.932,0.926,0.937,0.937
real_median_yes_no_mass,0.985,0.957,0.973,0.959,0.959,0.865,0.841,0.854,0.890,0.878,0.893,0.681,0.567
real_switch_ctr,0.501,0.488,0.490,0.484,0.527,0.525,0.522,0.499,0.477,0.483,0.515,0.528,0.515
real_switch_slope,33.051,29.730,32.282,32.022,40.886,38.614,36.902,31.986,24.607,24.339,35.927,31.977,25.532
transfer_phrasing_ctr_auroc,0.888,0.934,0.929,0.937,0.918,0.925,0.925,0.936,0.936,0.934,0.919,0.921,0.924
transfer_phrasing_brier,0.130,0.092,0.093,0.096,0.103,0.098,0.091,0.082,0.111,0.099,0.093,0.101,0.084
transfer_phrasing_nih_auroc,0.925,0.938,0.936,0.932,0.918,0.934,0.937,0.935,0.931,0.932,0.923,0.936,0.936
transfer_phrasing_median_yes_no_mass,0.991,0.964,0.972,0.955,0.937,0.756,0.763,0.848,0.884,0.875,0.862,0.597,0.478


model                                                base       R       E  \
metric                                       stat                           
real_ctr_auroc                               mean   0.900   0.937   0.925   
                                             min    0.900   0.934   0.920   
                                             max    0.900   0.940   0.927   
real_brier                                   mean   0.145   0.119   0.087   
                                             min    0.145   0.114   0.083   
                                             max    0.145   0.124   0.091   
real_nih_auroc                               mean   0.931   0.935   0.931   
                                             min    0.931   0.932   0.921   
                                             max    0.931   0.938   0.936   
real_median_yes_no_mass                      mean   0.985   0.963   0.888   
                                             min    0.985   0.957   0.841   
                                             max    0.985   0.973   0.959   
real_switch_ctr                              mean   0.501   0.487   0.525   
                                             min    0.501   0.484   0.522   
                                             max    0.501   0.490   0.527   
real_switch_slope                            mean  33.051  31.345  38.801   
                                             min   33.051  29.730  36.902   
                                             max   33.051  32.282  40.886   
transfer_phrasing_ctr_auroc                  mean   0.888   0.933   0.923   
                                             min    0.888   0.929   0.918   
                                             max    0.888   0.937   0.925   
transfer_phrasing_brier                      mean   0.130   0.094   0.097   
                                             min    0.130   0.092   0.091   
                                             max    0.130   0.096   0.103   
transfer_phrasing_nih_auroc                  mean   0.925   0.935   0.930   
                                             min    0.925   0.932   0.918   
                                             max    0.925   0.938   0.937   
transfer_phrasing_median_yes_no_mass         mean   0.991   0.964   0.819   
                                             min    0.991   0.955   0.756   
                                             max    0.991   0.972   0.937   
transfer_phrasing_switch_ctr                 mean   0.543   0.500   0.536   
                                             min    0.543   0.496   0.534   
                                             max    0.543   0.503   0.538   
transfer_phrasing_switch_slope               mean  35.922  33.205  37.024   
                                             min   35.922  31.648  34.437   
                                             max   35.922  34.115  40.062   
edited_brier                                 mean   0.303   0.138   0.074   
                                             min    0.303   0.121   0.069   
                                             max    0.303   0.148   0.077   
edited_ctr_auroc                             mean   0.941   0.968   0.970   
                                             min    0.941   0.964   0.969   
                                             max    0.941   0.971   0.972   
edited_within_growth_auroc                   mean   0.897   0.943   0.949   
                                             min    0.897   0.936   0.948   
                                             max    0.897   0.950   0.950   
edited_yes_when_ctr_below_05                 mean   0.499   0.282   0.087   
                                             min    0.499   0.232   0.071   
                                             max    0.499   0.327   0.095   
sham_yes                                     mean   0.190   0.163   0.047   
                                             min    0.190   0.160   0.040

## Every metric without the images within 0.025 of CTR 0.5

In [4]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(arms_without_margin.round(3))

model                                                base       R       E  \
metric                                       stat                           
real_ctr_auroc                               mean   0.950   0.979   0.969   
                                             min    0.950   0.978   0.968   
                                             max    0.950   0.981   0.970   
real_brier                                   mean   0.091   0.079   0.038   
                                             min    0.091   0.077   0.037   
                                             max    0.091   0.083   0.038   
real_nih_auroc                               mean   0.967   0.970   0.970   
                                             min    0.967   0.970   0.968   
                                             max    0.967   0.971   0.972   
real_median_yes_no_mass                      mean   0.986   0.972   0.914   
                                             min    0.986   0.965   0.877   
                                             max    0.986   0.980   0.966   
real_switch_ctr                              mean   0.504   0.491   0.526   
                                             min    0.504   0.488   0.524   
                                             max    0.504   0.493   0.527   
real_switch_slope                            mean  31.704  29.301  37.444   
                                             min   31.704  27.960  35.698   
                                             max   31.704  30.242  39.253   
transfer_phrasing_ctr_auroc                  mean   0.952   0.978   0.969   
                                             min    0.952   0.977   0.968   
                                             max    0.952   0.980   0.970   
transfer_phrasing_brier                      mean   0.067   0.052   0.042   
                                             min    0.067   0.051   0.040   
                                             max    0.067   0.054   0.044   
transfer_phrasing_nih_auroc                  mean   0.964   0.970   0.969   
                                             min    0.964   0.969   0.965   
                                             max    0.964   0.971   0.972   
transfer_phrasing_median_yes_no_mass         mean   0.993   0.972   0.848   
                                             min    0.993   0.962   0.798   
                                             max    0.993   0.979   0.946   
transfer_phrasing_switch_ctr                 mean   0.547   0.503   0.537   
                                             min    0.547   0.499   0.535   
                                             max    0.547   0.505   0.537   
transfer_phrasing_switch_slope               mean  35.989  31.606  36.001   
                                             min   35.989  30.130  33.637   
                                             max   35.989  32.722  38.739   
edited_brier                                 mean   0.221   0.056   0.020   
                                             min    0.221   0.043   0.019   
                                             max    0.221   0.067   0.023   
edited_ctr_auroc                             mean   0.987   0.995   0.997   
                                             min    0.987   0.994   0.997   
                                             max    0.987   0.997   0.998   
edited_within_growth_auroc                   mean   0.965   0.986   0.994   
                                             min    0.965   0.980   0.992   
                                             max    0.965   0.990   0.994   
edited_yes_when_ctr_below_05                 mean   0.356   0.116   0.024   
                                             min    0.356   0.080   0.019   
                                             max    0.356   0.144   0.027   
sham_yes                                     mean   0.105   0.097   0.035   
                                             min    0.105   0.093   0.035

## Paired patient bootstrap

1,000 resamples of patients, separately for the real films and for the edits' source patients, the
same for every model. Contrasts: R's, E's and RE's real-film CTR-AUROC gains, E's edited-film Brier
gain, D = R's real gain - 2 x E's real gain, RE - R (CTR-AUROC; secondary: NIH AUROC) and Brier(R) -
Brier(RE) (secondary; positive favours RE), and the effusion AUROC drop of each arm (guard).

In [5]:
rng = np.random.default_rng(0)
real_pat = base.loc[real_idx, "patient_id"].to_numpy()
edit_pat = base.loc[edit_idx, "patient_id"].to_numpy()
groups_r = pd.Series(np.arange(len(real_idx))).groupby(real_pat).apply(list).tolist()
groups_e = pd.Series(np.arange(len(edit_idx))).groupby(edit_pat).apply(list).tolist()
D = {n: {"y_r": tables[n].loc[real_idx, "label"].to_numpy(), "d_r": tables[n].loc[real_idx, "log_odds"].to_numpy(),
         "p_r": tables[n].loc[real_idx, "p_yes"].to_numpy(), "y_e": tables[n].loc[edit_idx, "label"].to_numpy(),
         "p_e": tables[n].loc[edit_idx, "p_yes"].to_numpy()} for n in names}
nih = tables["base"].loc[real_idx, ["cardiomegaly", "no_finding", "effusion"]].to_numpy()
real_images = base.loc[real_idx, "image"].to_numpy()
for n in names:  # the effusion question, aligned with the real films
    D[n]["d_eff"] = table(n, "effusion_0").set_index("image").loc[real_images, "log_odds"].to_numpy()

def stats(ir, ie):
    """Per model: real CTR-AUROC, real Brier, real NIH AUROC, edited Brier and effusion AUROC
    (effusion vs no finding), on resampled indices."""
    res = {}
    keep = (nih[ir, 0] == 1) | (nih[ir, 1] == 1)
    keep_eff = (nih[ir, 2] == 1) | (nih[ir, 1] == 1)
    for n in names:
        x = D[n]
        res[n] = (roc_auc_score(x["y_r"][ir], x["d_r"][ir]), ((x["p_r"][ir] - x["y_r"][ir]) ** 2).mean(),
                  roc_auc_score(nih[ir, 0][keep], x["d_r"][ir][keep]), ((x["p_e"][ie] - x["y_e"][ie]) ** 2).mean(),
                  roc_auc_score(nih[ir, 2][keep_eff], x["d_eff"][ir][keep_eff]))
    return res

def contrasts(res, seeds=SEEDS):
    m = lambda arm, k: np.mean([res[f"{arm}{s}"][k] for s in seeds])
    b = res["base"]
    gain = {a: m(a, 0) - b[0] for a in ARMS}
    return {"gain_R_real": gain["R"], "gain_E_real": gain["E"], "gain_RE_real": gain["RE"],
            "gain_E_edited": b[3] - m("E", 3), "D": gain["R"] - 2 * gain["E"], "RE_minus_R_auroc": m("RE", 0) - m("R", 0),
            "R_minus_RE_brier": m("R", 1) - m("RE", 1), "RE_minus_R_nih_auroc": m("RE", 2) - m("R", 2),
            **{f"effusion_drop_{a}": b[4] - m(a, 4) for a in ARMS}}

full = contrasts(stats(np.arange(len(real_idx)), np.arange(len(edit_idx))))
boots = []
for _ in range(1000):
    ir = np.concatenate([groups_r[i] for i in rng.integers(len(groups_r), size=len(groups_r))])
    ie = np.concatenate([groups_e[i] for i in rng.integers(len(groups_e), size=len(groups_e))])
    boots.append(contrasts(stats(ir, ie)))
boots = pd.DataFrame(boots)
ci = pd.DataFrame({"point": full, "ci_low": boots.quantile(0.025), "ci_high": boots.quantile(0.975)})
per_seed = pd.DataFrame([contrasts(stats(np.arange(len(real_idx)), np.arange(len(edit_idx))), seeds=[k]) for k in SEEDS], index=SEEDS)
ci.round(4)

,point,ci_low,ci_high
gain_R_real,0.0373,0.0254,0.0498
gain_E_real,0.0250,0.0150,0.0349
gain_RE_real,0.0368,0.0233,0.0499
gain_E_edited,0.2290,0.1908,0.2679
D,-0.0126,-0.0272,0.0023
RE_minus_R_auroc,-0.0005,-0.0052,0.0038
R_minus_RE_brier,0.0003,-0.0057,0.0060
RE_minus_R_nih_auroc,-0.0025,-0.0064,0.0014
effusion_drop_R,-0.0103,-0.0155,-0.0044
effusion_drop_E,-0.0025,-0.0072,0.0021


In [6]:
per_seed.round(4)

,gain_R_real,gain_E_real,gain_RE_real,gain_E_edited,D,RE_minus_R_auroc,R_minus_RE_brier,RE_minus_R_nih_auroc,effusion_drop_R,effusion_drop_E,effusion_drop_RE
0,0.0377,0.0203,0.0369,0.2271,-0.0029,-0.0007,0.0252,-0.0026,-0.0098,-0.0032,-0.0109
1,0.0344,0.0276,0.0372,0.2259,-0.0208,0.0028,-0.0220,-0.0048,-0.0106,-0.0020,-0.0092
2,0.0398,0.0270,0.0363,0.2341,-0.0142,-0.0035,-0.0024,-0.0002,-0.0103,-0.0023,-0.0087


## Robustness checks for H3.1 (point estimates of D)

In [7]:
def d_point(metric_fn):
    gain = lambda arm, real=False: np.mean([metric_fn(f"{arm}{s}" if not real else f"E{s}real") for s in SEEDS]) - metric_fn("base")
    return gain("R") - 2 * gain("E")

def auroc_with(label_col):
    def f(name):
        t = tables[name].loc[real_idx]
        t = t[t[label_col].notna()]
        return roc_auc_score(t[label_col] > 0.5, t["log_odds"])
    return f

def stratified(name):
    """CTR-AUROC over positive-negative pairs within NIH finding status only (no finding / any finding)."""
    t = tables[name].loc[real_idx]
    num = den = 0.0
    for _, g in t.groupby(t["no_finding"] == 1):
        if 0 < g["label"].sum() < len(g):
            w = g["label"].sum() * (len(g) - g["label"].sum())
            num, den = num + w * roc_auc_score(g["label"], g["log_odds"]), den + w
    return num / den

ctr_auroc = auroc_with("ctr")
real_gain_E_best_real = np.mean([ctr_auroc(f"E{s}real") for s in SEEDS]) - ctr_auroc("base")
robust = {"D with the CheXmask label": d_point(auroc_with("ctr_chexmask")),
          "D within NIH finding status": d_point(stratified),
          "D with E at its best real-validation checkpoint": full["gain_R_real"] - 2 * real_gain_E_best_real}
pd.Series(robust).round(4)

D with the CheXmask label                         -0.0492
D within NIH finding status                       -0.0046
D with E at its best real-validation checkpoint   -0.0122
dtype: float64

## Verdicts

Rules as pre-registered, with the clarifications of 2026-10-01 (PLAN.md, H3); "H3.1 supported" also
needs E's edited-film gain to be positive for every seed, a clarification added during the training
session (12:12 UTC, commit 1cf72fe).

In [8]:
bar_real = 0.2 * (1 - point.loc["base", "real_ctr_auroc"])
bar_edit = 0.2 * point.loc["base", "edited_brier"]
c = ci.T
moves = lambda k, bar: c[k]["point"] >= bar and c[k]["ci_low"] > 0
does_not_move = lambda k, bar: c[k]["ci_high"] < bar
seeds_agree = lambda k, sign: bool((np.sign(per_seed[k]) == sign).all())
precondition = moves("gain_R_real", bar_real)
if not precondition:
    h31 = "inconclusive: R's real-film gain does not move the model (precondition)"
elif (moves("gain_E_edited", bar_edit) and seeds_agree("gain_E_edited", 1)  # both deciding contrasts: every seed
      and c["D"]["ci_low"] > 0 and seeds_agree("D", 1) and all(v > 0 for v in robust.values())):
    h31 = "supported"
else:
    ways = []  # both named ways are reported when both hold
    if does_not_move("gain_E_edited", bar_edit) and bool((per_seed["gain_E_edited"] < bar_edit).all()):
        ways.append("E did not learn the edits")
    if c["D"]["ci_high"] < 0 and seeds_agree("D", -1):
        ways.append("E transfers (more than half of R's real-film gain)")
    h31 = "falsified: " + "; ".join(ways) if ways else "inconclusive"
gain_ratio = c["gain_E_real"]["point"] / c["gain_R_real"]["point"] if c["gain_R_real"]["point"] != 0 else np.nan
def h32_verdict(k):
    """H3.2's rule and 0.02 margin applied to contrast k (positive favours RE)."""
    x = c[k]
    if x["ci_low"] > 0 and x["point"] >= 0.02 and seeds_agree(k, 1):
        return "falsified: RE's gain over R is at least the 0.02 margin"
    if not precondition:
        return "inconclusive: precondition not met (only the falsification is evaluated)"
    if x["ci_high"] < 0.02 and bool((per_seed[k] < 0.02).all()):
        return "supported: no gain of RE over R as large as the 0.02 margin"
    return "inconclusive"

h32 = h32_verdict("RE_minus_R_auroc")
h32_secondary = {"Brier, Brier(R) - Brier(RE)": h32_verdict("R_minus_RE_brier"),
                 "NIH-label AUROC, RE - R": h32_verdict("RE_minus_R_nih_auroc")}
guards = {f"{a}: blank yes (prior-corrected) > 5%": bool(np.mean([point.loc[f"{a}{s}", "blank_yes_prior_corrected"] for s in SEEDS]) > 0.05)
          for a in ARMS}
guards.update({f"{a}: effusion AUROC drop > 0.02, CI above 0": bool(c[f"effusion_drop_{a}"]["point"] > 0.02 and c[f"effusion_drop_{a}"]["ci_low"] > 0)
               for a in ARMS})
verdicts = pd.DataFrame([{"set": SET, "dry_run": DRY_RUN, "bar_real": bar_real, "bar_edited": bar_edit,
                          "precondition": precondition, "H3.1": h31, "H3.2": h32,
                          "ratio of real-film gains, E / R (description)": gain_ratio,
                          **{f"H3.2 secondary: {k}": v for k, v in h32_secondary.items()},
                          **{f"guard: {k}": v for k, v in guards.items()}, **{f"robustness: {k}": v for k, v in robust.items()}}])
if not DRY_RUN:
    verdicts.to_csv(R / f"h3_verdicts_{SET}.csv", index=False)
    ci.to_csv(R / f"h3_contrasts_{SET}.csv")
    per_seed.rename_axis("seed").to_csv(R / f"h3_per_seed_{SET}.csv")  # the contrasts with seed k of every arm
verdicts.T

,0
set,fresh
dry_run,False
bar_real,0.020035
bar_edited,0.060519
precondition,True
H3.1,inconclusive
H3.2,supported: no gain of RE over R as large as th...
"ratio of real-film gains, E / R (description)",0.669561
"H3.2 secondary: Brier, Brier(R) - Brier(RE)",inconclusive
"H3.2 secondary: NIH-label AUROC, RE - R",supported: no gain of RE over R as large as th...


## Takeaways

In [9]:
if DRY_RUN:
    print("DRY RUN: arms without scores were replaced by the base model; these verdicts only test the code path,"
          " and no metrics, contrasts or verdicts table was written to results/.")
print(f"Set {SET}: bars {bar_real:.3f} (real CTR-AUROC) and {bar_edit:.3f} (edited Brier).")
for k in ["gain_R_real", "gain_E_real", "gain_RE_real", "gain_E_edited", "D", "RE_minus_R_auroc"]:
    print(f"  {k}: {c[k]['point']:+.4f} (95% CI {c[k]['ci_low']:+.4f} to {c[k]['ci_high']:+.4f}); per seed {per_seed[k].round(4).tolist()}")
print(f"  ratio of real-film gains, E / R (description): {gain_ratio:.3f}")
print("H3.1:", h31)
print("H3.2:", h32)
for k, v in h32_secondary.items():
    print(f"H3.2 secondary ({k}): {v}")

Set fresh: bars 0.020 (real CTR-AUROC) and 0.061 (edited Brier).
  gain_R_real: +0.0373 (95% CI +0.0254 to +0.0498); per seed [0.0377, 0.0344, 0.0398]
  gain_E_real: +0.0250 (95% CI +0.0150 to +0.0349); per seed [0.0203, 0.0276, 0.027]
  gain_RE_real: +0.0368 (95% CI +0.0233 to +0.0499); per seed [0.0369, 0.0372, 0.0363]
  gain_E_edited: +0.2290 (95% CI +0.1908 to +0.2679); per seed [0.2271, 0.2259, 0.2341]
  D: -0.0126 (95% CI -0.0272 to +0.0023); per seed [-0.0029, -0.0208, -0.0142]
  RE_minus_R_auroc: -0.0005 (95% CI -0.0052 to +0.0038); per seed [-0.0007, 0.0028, -0.0035]
  ratio of real-film gains, E / R (description): 0.670
H3.1: inconclusive
H3.2: supported: no gain of RE over R as large as the 0.02 margin
H3.2 secondary (Brier, Brier(R) - Brier(RE)): inconclusive
H3.2 secondary (NIH-label AUROC, RE - R): supported: no gain of RE over R as large as the 0.02 margin
